# Quantification of Mitochondria in 2D Fluorescent Stacks
- *Isobel Taylor-Hearn, 2023*
- Requires a fluorescent image stack containing (at least) a nuclear channel and mitochondria channel

In [56]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import math
import os
import pathlib
from liffile import LifFile
import tifffile
import skimage
from skimage.feature import peak_local_max
from skimage import util,  restoration
from skimage.filters import threshold_otsu, threshold_isodata
from skimage.segmentation import clear_border, expand_labels, watershed
from skimage.measure import label, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, closing,  disk, dilation
from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
import warnings
warnings.filterwarnings("ignore")
import pathlib

import contextlib
import joblib
from tqdm import tqdm
from joblib import Parallel, delayed

In [57]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """
    Enables parallel jobs to run and display of a tqdm progress bar.

    Parameters:
    -----------
    tqdm_object : tqdm
        The tqdm progress bar instance to be updated.

    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [58]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [59]:
def add_image_details(df, filename):
    """
    Adds experimental details extracted from the filename to a dataframe.

    This function parses the filename to infer experimental details such as 
    well number, imaging day, mechanical stiffness condition, and treatment type.
    The extracted details are appended as new columns to the dataframe.

    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to which image metadata will be added.
    filename : str
        The filename of the image, used to extract experimental details.

    Returns:
    --------
    df : pandas.DataFrame
        The updated dataframe with the following added columns:
        - 'filename': The original filename.
        - 'flag': Segmentation flag indicating potential issues.
        - 'condition': WT, Nic, Tom or Nictom (or unknown if no match found)
    """

    df["image"] = filename
    if "10a" in str(filename).lower():
        df["cell_type"] = "MCF10A"
    else:
        df["cell_type"] = "OTHER"
    if "4oht" in str(filename).lower():
        df["treatment"] = "4OHT" 
    elif "abt" in str(filename).lower():
        df["treatment"] = "ABT737"  
    else:
        df["treatment"] = "OTHER"  
    return df

In [60]:
def segment_nuclei(dapi, pixel_size):
    """
    Segment nuclei and use a watershed segmentation to separate touching objects.
    Filter out any objects below a defined area cutoff.
    Acquired images did not have a membrane marker. Therefore, the code assumes that the border between two cells lies halfway between
        the shortest distance between their nuclei.
    Cell bodies are therefore approximated by expanding nuclear labels without overlap until they fill the image. Cells that touch the border
        or contain nuclei larger than a 15 µm radius equivalent are removed from the analysis.

    Parameters
    ----------
    dapi : ndarray
        Grayscale image of nuclei stained with DAPI.
    pixel_size : float
        Pixel size in microns (µm), used to filter out overly large nuclei.

    Returns
    -------
    dapi_labels : ndarray (int)
        Labeled image of segmented nuclei, where each nucleus has a unique label.
    valid_cell_labels : ndarray (int)
        Labeled image of cell regions corresponding to valid cells. Cells are valid if they don't touch the image border and
            have nuclei smaller than a 15 µm radius equivalent.
    """
    ################## Segment all nuclei #######################################################################################################
    print(pixel_size)
    thresh = threshold_otsu(dapi)
    dapi_processed = dapi > thresh
    dapi_processed = closing(dapi_processed, disk(5))
    dapi_processed = remove_small_holes(dapi_processed, area_threshold = 10000)
    dapi_processed = remove_small_objects(dapi_processed, min_size=1000)
    dapi_labels = label(dapi_processed)
    distances = ndi.distance_transform_edt(dapi_processed)
    coordinates = peak_local_max(distances, min_distance = int(5*pixel_size)) # min_distance = min separation of nuclear COMs
    marker_locations = coordinates.data
    markers = np.zeros(dapi_processed.shape, dtype=np.uint32)
    marker_indices = tuple(np.round(marker_locations).astype(int).T)
    markers[marker_indices] = np.arange(len(marker_locations)) + 1
    markers_big = dilation(markers, disk(5))
    dapi_labels_ws = watershed(-distances, markers_big, mask=dapi_processed)
    dapi_labels = label(dapi_labels_ws + dapi_labels)
    ################## Approximate cell boundaries and exclude cells that touch the border from analysis #########################################
    all_cells_labels = clear_border(expand_labels(dapi_labels, 1000)) # any cells that don't touch the border
    nuclear_props = regionprops_table(dapi_labels, dapi, properties=("label", "area"))  
    condition = (nuclear_props['area'] <= np.pi *((15*pixel_size)**2)) # filter out any cells with nuclei more than 15um radius equivalent
    input_labels = nuclear_props['label']
    output_labels = input_labels * condition
    valid_cell_labels = util.map_array(all_cells_labels, input_labels, output_labels) # only keep the cells with nuclei that pass the area test
    return dapi_labels, valid_cell_labels

In [61]:
def segment_mitochondria(mito, background_subtraction):
    """
    Segment mitochondria from a grayscale fluorescence image.

    Parameters
    ----------
    mito : ndarray
        Grayscale image of mitochondria, where mitochondria appear bright on a dark background.
    background_subtraction : bool
        If True, applies rolling ball background subtraction before segmentation.

    Returns
    -------
    binary_mito : ndarray (bool)
        Binary mask of segmented mitochondria.
    mito_labels : ndarray (int)
        Labeled image of mitochondria, where each connected component has a unique integer label.
    """
    if background_subtraction == True:
        print("Background Subtraction Applied")
        background = restoration.rolling_ball(mito, radius = 11)
        mito_processed = mito - background
    else:
        mito_processed = mito
    thresh=threshold_isodata(mito_processed)
    binary_mito = mito_processed > thresh
    mito_labels = label(binary_mito)
    return binary_mito, mito_labels

In [62]:
def quantify_mito_shape(cell_label, binary_mito, cell_labels, pixel_size):
    # Restrict the mito mask to this cell before labelling, so objects straddling a
    # cell boundary are split there and each piece is counted for its own cell.
    lbl = label(binary_mito & (cell_labels == cell_label))
    if lbl.max() == 0:
        return pd.Series({"n_mito_objects": 0, "mean_mito_length_um": np.nan,
                            "std_mito_length_um": np.nan, "mean_mito_aspect_ratio": np.nan,
                            "mean_mito_area_um": np.nan})
    props = pd.DataFrame(regionprops_table(lbl, properties=("area", "major_axis_length", "minor_axis_length")))
    length_um = props["major_axis_length"] * (1 / pixel_size)
    aspect = props["major_axis_length"] / props["minor_axis_length"].replace(0, np.nan)
    return pd.Series({
        "n_mito_objects": int(len(props)),
        "mean_mito_length_um": length_um.mean(),
        "std_mito_length_um": length_um.std(),
        "mean_mito_aspect_ratio": aspect.mean(),
        "mean_mito_area_um": (props["area"] * ((1 / pixel_size) ** 2)).mean(),
    })

In [63]:

def segment_nuclei_and_mitochondria(image, filename, axes="CZYX", to_plot=True, dapi_channel=0, mito_channel=1, pixel_size=7.4627, background_subtraction=False, analyse_distance=False, project=True):
    """
    Segment nuclei and mitochondria from a multi-channel image and quantify mitochondrial content per cell.

    Nuclei and cells are segmented from the DAPI channel and mitochondria from the mitochondrial
    fluorescence channel. Per-cell mitochondrial area, morphology (punctate vs. connected) and,
    optionally, the radial distribution of mitochondria relative to the nucleus are quantified.

    Parameters
    ----------
    image : ndarray
        Multi-channel microscopy image (any axis order, described by `axes`).
    filename : str
        Image name, used to annotate the output tables.
    axes : str, optional (default="CZYX")
        Axis order of `image` (case-insensitive). Used to project over Z and locate the channel axis.
    to_plot : bool, optional (default=True)
        Whether to generate a plot showing the segmentation steps. Keep False when running in parallel.
    dapi_channel : int, optional (default=0)
        Channel index for DAPI (nuclear stain).
    mito_channel : int, optional (default=1)
        Channel index for mitochondrial fluorescence.
    pixel_size : float, optional (default=7.4627)
        Pixels per micron (i.e. 1 / µm-per-pixel). Should be supplied from the image metadata.
    background_subtraction : bool, optional (default=False)
        Whether to apply rolling ball background subtraction to the mitochondrial image before segmentation.
    analyse_distance : bool, optional (default=False)
        Whether to compute the mitochondrial distribution as a function of distance from the nucleus.
    project : bool, optional (default=True)
        Whether to max-project over the Z axis (images with a Z dimension should use True).

    Returns
    -------
    mito_area_df : pandas.DataFrame
        Per-cell summary table containing:
        - 'image', 'cell_type', 'treatment', 'cell_label'
        - 'cell_area_um', 'mito_area_um', 'mito/cell_area'
        - 'n_mito_objects'          : number of separate mitochondrial objects in the cell
        - 'mean_mito_length_um'     : mean major-axis length of mitochondrial objects (µm)
        - 'std_mito_length_um'      : std of the major-axis lengths (µm)
        - 'mean_mito_aspect_ratio'  : mean major/minor axis ratio (1 = round/punctate, >>1 = elongated/connected)
        - 'mean_mito_area_um'       : mean area of a mitochondrial object (µm²)
    mito_distance_df : pandas.DataFrame
        Pixel-level table of mitochondrial presence vs. distance from the nucleus (empty if
        `analyse_distance` is False or no valid cells are found).
    """
    axes = list(axes.upper())
    sum_mito = False
    if "Z" in axes:
        image = np.max(image, axis=axes.index("Z"))
        sum_mito = image[mito_channel, :, :]   # keep the projected mito image, don't sum it
    dapi = convert(image[dapi_channel, :, :], 0, 255, np.uint8)
    mito = convert(image[mito_channel, :, :], 0, 255, np.uint8)

    rgb_image = np.stack([np.zeros_like(dapi), mito, dapi], axis=-1)

    dapi_labels, cell_labels = segment_nuclei(dapi, pixel_size=pixel_size)
    binary_mito, mito_labels = segment_mitochondria(mito, background_subtraction)
    
    if to_plot == True:
        fig, ax = plt.subplots(ncols = 4, figsize=(20,5))
        ax[0].imshow(rgb_image)
        ax[1].imshow(dapi, alpha = 0.7, cmap="gray", interpolation = "none")
        ax[1].imshow(np.ma.masked_where(dapi_labels == 0, dapi_labels), cmap="jet", alpha=0.5, interpolation="none")
        ax[1].imshow(np.ma.masked_where(cell_labels == 0, cell_labels), cmap="jet", alpha=0.5, interpolation="none")
        ax[2].imshow(mito, alpha = 0.7, cmap="gray", interpolation = "none")
        ax[2].imshow(np.ma.masked_where(mito_labels == 0, mito_labels), cmap="jet", alpha=0.5, interpolation="none")
        ax[3].imshow(rgb_image)
        ax[3].imshow(np.ma.masked_where((dapi_labels == 0) | (cell_labels == 0), dapi_labels), alpha=0.5, cmap="gray", interpolation="none")
        ax[3].imshow(np.ma.masked_where((mito_labels == 0) | (cell_labels == 0), mito_labels), alpha=0.8, cmap="jet", interpolation="none")
        
        
        cell_boundaries = find_boundaries(cell_labels, mode="outer")
        # ax[4].imshow(np.ma.masked_where(cell_labels_paired == 0, cell_labels_paired), cmap="jet", alpha=0.3, interpolation="none")
        ax[1].imshow(np.ma.masked_where(~cell_boundaries, cell_labels), cmap="gray", zorder=1000)
    
        for a in ax:
            a.axis("off")

        plt.savefig(os.path.join(out_dir, f"{filename}_segmentation.png"))
################## Basic Quantification #######################################################################################################
    mito_area_df = pd.DataFrame(regionprops_table(cell_labels, dapi, properties=("label", "area")) )
    if sum_mito is not False:
        mito_area_df["sum_mito"] = mito_area_df.apply(lambda row: sum_mito[cell_labels == row["label"]].sum(), axis=1)
    mito_area_df["mito/cell_area"]= mito_area_df.apply(lambda row: binary_mito[cell_labels == row["label"]].sum() / row["area"], axis=1)
    mito_area_df["mito_area_um"] = mito_area_df.apply(lambda row: (binary_mito[cell_labels == row["label"]].sum())* ((1/pixel_size)**2), axis=1)
    mito_area_df["cell_area_um"] = mito_area_df["area"] * ((1/pixel_size)**2)

################## Mitochondrial Shape (punctate vs. connected) ################################################################################

    shape_df = mito_area_df["label"].apply(quantify_mito_shape, args=(binary_mito, cell_labels, pixel_size))
    mito_area_df = pd.concat([mito_area_df, shape_df], axis=1)

    mito_area_df = add_image_details(mito_area_df, filename)
    mito_area_df = mito_area_df.rename(columns={'label': 'cell_label'})
    mito_area_df = mito_area_df[['image', 'cell_type', 'treatment', 'cell_label', 'cell_area_um', 'mito_area_um', 'mito/cell_area',
                                 'n_mito_objects', 'mean_mito_length_um', 'std_mito_length_um', 'mean_mito_aspect_ratio', 'mean_mito_area_um', "sum_mito"]]

################## Mitochondria Distance from Nucleus Analysis ##################################################################################
    if analyse_distance:
        try: # if full cells are identified
            mito_distance_df = []
            for i in np.unique(cell_labels):
                if i != 0:       
                    nuc_of_interest = dapi_labels * (dapi_labels == i) > 0 
                    cell_of_interest = cell_labels * (cell_labels == i) > 0 
                    distance = ((distance_transform_edt(1- (nuc_of_interest))) * cell_of_interest) * (1/pixel_size)
                    binary_mito_in_cell = cell_of_interest * binary_mito
                    count_with_distance = pd.DataFrame(zip(np.ravel(distance), np.ravel(binary_mito_in_cell)), columns=['distance', 'count'])   
                    count_with_distance["cell_label"] = i
                    mito_distance_df.append(count_with_distance) # looks at every pixel at every distance and evaluates is this pixel filled by mitochondria
            mito_distance_df = pd.concat(mito_distance_df).reset_index()
            mito_distance_df = mito_distance_df[mito_distance_df.distance != 0]
            mito_distance_df['rounded_distance'] = mito_distance_df['distance'].apply(lambda x: round(x, 2))
            mito_distance_df = add_image_details(mito_distance_df, filename)

            mito_distance_df = mito_distance_df[['image', 'cell_type', 'treatment', 'cell_label', 'rounded_distance', 'count']]
        except:
            mito_distance_df = pd.DataFrame()
    else:
        mito_distance_df = pd.DataFrame()
    return (mito_area_df, mito_distance_df)


In [64]:
def plot_outputs(df, max_distance_quantile=50):
    """
    Plot mitochondrial distribution as a function of distance from the nucleus for each cell.

    This function aggregates and smooths pixel-level mitochondrial occupancy data per cell. It filters
    distances to include only those observed in at least a specified percentage of cells, then applies a 
    rolling average to smooth the data and generates two plots:
    1. Proportion of pixels at a given distance that contain mitochondria
    2. Absolute number of mitochondrial pixels at each distance

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing columns:
        - 'cell_label': int, unique ID per cell
        - 'rounded_distance': float, distance from nucleus in microns
        - 'count': int, binary indicator of whether pixel is occupied by mitochondria (0 or 1)

    max_distance_quantile : float, optional (default=50)
        Distance cutoff threshold. Only distances observed in at least this percentage of cells are included
        in the analysis.

    Returns
    -------
    smoothed : pandas.DataFrame
        Smoothed and filtered DataFrame with per-cell mitochondrial occupancy statistics.
    """


    df["count"] = df["count"].astype(int)
    df = (df.groupby(["cell_label", "rounded_distance"])["count"].agg(["sum", "count"]).rename(columns={
            "sum": "number_of_mito_at_distance",
            "count": "number_of_pixels_at_distance"
        }).reset_index())
    df["proportion_of_pixels_with_mito"] = (df["number_of_mito_at_distance"] / df["number_of_pixels_at_distance"])

    # Keep only distances present in >= max_distance_quantile% of cells
    distance_counts = df.groupby("rounded_distance")["cell_label"].nunique()
    threshold = (max_distance_quantile / 100) * df["cell_label"].nunique()
    valid_distances = distance_counts[distance_counts >= threshold].index
    df = df[df["rounded_distance"].isin(valid_distances)].copy()
    smoothed = (
        df.sort_values(["cell_label", "rounded_distance"])
        .groupby("cell_label")
        .rolling(window=55, center=True, min_periods=1, on="rounded_distance")
        .mean(numeric_only=True)
        .reset_index()
    )
    smoothed["cell_label"] = smoothed["cell_label"].astype(int)

    fig, ax = plt.subplots(ncols=2, figsize=(15,6))
    sns.lineplot(data=smoothed, x="rounded_distance", y="proportion_of_pixels_with_mito", hue="cell_label", palette="gist_rainbow", ax=ax[0])
    sns.lineplot(data=smoothed, x="rounded_distance", y="number_of_mito_at_distance", hue="cell_label", palette="gist_rainbow", ax=ax[1])
    for a in ax:
        a.set_xlabel("Distance from Nucleus (µm)")
    ax[0].set_ylabel("Proportion of Pixels Occupied by Mitochondria")
    ax[1].set_ylabel("Number of Mitochondrial Pixels")

    return smoothed

In [65]:
def _pixel_size_from_tif(tif):
    """Return the pixel size of an open TiffFile as pixels-per-micron, or None if unknown."""
    page = tif.pages[0]
    ij = tif.imagej_metadata or {}
    unit = str(ij.get("unit", "")).lower()
    xres = page.tags.get("XResolution")
    if xres is None:
        return None
    num, den = xres.value
    if den == 0:
        return None
    pixels_per_unit = num / den  # pixels per resolution unit
    if unit in ("um", "micron", "microns", "µm", "\\u00b5m"):
        return pixels_per_unit                     # already pixels per micron
    runit = page.tags.get("ResolutionUnit")
    if runit is not None and runit.value == 3:      # centimetre
        return pixels_per_unit / 10_000.0           # px/cm -> px/µm
    if runit is not None and runit.value == 2:      # inch
        return pixels_per_unit / 25_400.0           # px/inch -> px/µm
    return None


def load_tif_image(path):
    """Load a TIF image, returning (data, axes, pixels_per_micron)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        data = series.asarray()
        axes = series.axes
        pxpum = _pixel_size_from_tif(tif)
    return data, axes, pxpum


def load_lif_image(path, subpath):
    """Load one image from a LIF file, returning (data, axes, pixels_per_micron, name)."""
    with LifFile(path) as lif:
        img = next(im for im in lif.images if "".join(im.path) == subpath)
        xa = img.asxarray()
        data = np.asarray(xa)
        axes = "".join(str(d) for d in xa.dims).upper()
        pxpum = None
        coords = xa.coords
        if "X" in coords and coords["X"].size >= 2:
            x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6  # µm per pixel
            if x_um > 0:
                pxpum = 1.0 / x_um                                    # pixels per µm
    name = subpath.replace("/", "_")
    return data, axes, pxpum, name


def process_task(task, to_plot=False, dapi_channel=0, mito_channel=1,
                 background_subtraction=True, analyse_distance=False, project=True, out_dir = r"./"):
    """
    Load and analyse a single image described by `task` = (source, path, subname).

    `source` is "tif" (subname ignored) or "lif" (subname = image path within the .lif).
    The pixel size is always read from the file metadata; images without a readable
    pixel size are skipped. Returns (mito_area_df, mito_distance_df).
    """
    source, path, subname = task
    try:
        if source == "tif":
            data, axes, pxpum = load_tif_image(path)
            name = os.path.basename(path).split(".")[0]
        else:
            data, axes, pxpum, name = load_lif_image(path, subname)

        if pxpum is None:
            print(f"[WARN] No pixel size in metadata for {name}; skipping.")
            return (pd.DataFrame(), pd.DataFrame())

        return segment_nuclei_and_mitochondria(
            data, name, axes=axes, pixel_size=pxpum, to_plot=to_plot,
            dapi_channel=dapi_channel, mito_channel=mito_channel,
            background_subtraction=background_subtraction,
            analyse_distance=analyse_distance, project=project,
        )
    except Exception as e:
        print(f"[ERROR] {source} {path} {subname}: {type(e).__name__}: {e}")
        return (pd.DataFrame(), pd.DataFrame())


In [66]:
root = r"Z:\Bel\Chiara_Mito\in"
tif_paths = list(pathlib.Path(root).glob("**/*.tif"))
lif_paths = list(pathlib.Path(root).glob("**/*.lif"))
out_dir = r"Z:\Bel\Chiara_Mito\out"
os.makedirs(out_dir, exist_ok=True)
# Build a flat task list. Each TIF is one image; each LIF contributes one task per image it contains.
tasks = []
for p in tif_paths:
    tasks.append(("tif", str(p), None))
for p in lif_paths:
    with LifFile(p) as lif:
        for img in lif.images:
            tasks.append(("lif", str(p), "".join(img.path)))

print(f"{len(tif_paths)} TIF file(s), {len(lif_paths)} LIF file(s) -> {len(tasks)} image(s) to analyse")


0 TIF file(s), 1 LIF file(s) -> 60 image(s) to analyse


In [67]:
# Preview a single image to confirm the Z max-projection and channel assignment look correct.
# `project=True` because these stacks contain a Z dimension; the pixel size comes from the metadata.
(mito_area_df, mito_distance_df) = process_task(
    tasks[0], to_plot=True, dapi_channel=0, mito_channel=1,
    background_subtraction=True, analyse_distance=False, project=True, out_dir = out_dir
)
mito_area_df


3.5200000000000005
Background Subtraction Applied


,image,cell_type,treatment,cell_label,cell_area_um,mito_area_um,mito/cell_area,n_mito_objects,mean_mito_length_um,std_mito_length_um,mean_mito_aspect_ratio,mean_mito_area_um,sum_mito
0,40PFOA_A3_P 1,OTHER,OTHER,7,4772.565857,216.054365,0.045270,131.0,1.388539,2.158164,2.416251,1.649270,477055
1,40PFOA_A3_P 1,OTHER,OTHER,12,5137.687242,402.004778,0.078246,236.0,1.506365,2.948833,2.549678,1.703410,678473
2,40PFOA_A3_P 1,OTHER,OTHER,13,3374.386622,349.383394,0.103540,186.0,1.714317,3.002719,2.562353,1.878405,615734
3,40PFOA_A3_P 1,OTHER,OTHER,14,6467.507102,549.941890,0.085032,218.0,2.002420,4.610408,2.724483,2.522669,902127
4,40PFOA_A3_P 1,OTHER,OTHER,15,4751.904700,543.888817,0.114457,118.0,1.524213,6.021641,2.662646,4.609227,950950
5,40PFOA_A3_P 1,OTHER,OTHER,17,5334.613895,330.739928,0.061999,107.0,1.502978,4.492205,2.128808,3.091027,659574


In [68]:
# smoothed = plot_outputs(mito_distance_df)

# Batch Analysis of All Files (TIF + LIF)


In [69]:
# Analyse every TIF and LIF image in parallel (one worker per CPU core).
# Plotting is disabled in the workers; set analyse_distance=True to also collect radial profiles.
with tqdm_joblib(tqdm(desc="Image Analysis", total=len(tasks))) as progress_bar:
    output = Parallel(n_jobs=-1)(
        delayed(process_task)(t, to_plot=True, background_subtraction=True, analyse_distance=False, project=True)
        for t in tasks
    )

total_area = pd.concat([out[0] for out in output], ignore_index=True)
total_area.to_csv("mito_area_data.csv", index=False)

total_dist = pd.concat([out[1] for out in output], ignore_index=True)
total_dist.to_csv("mito_distance_data.csv", index=False)

print(f"Analysed {len(tasks)} image(s); {len(total_area)} cell(s) quantified.")
total_area.head()


Image Analysis: 100%|██████████| 60/60 [07:09<00:00,  7.17s/it]  

Analysed 60 image(s); 982 cell(s) quantified.


,image,cell_type,treatment,cell_label,cell_area_um,mito_area_um,mito/cell_area,n_mito_objects,mean_mito_length_um,std_mito_length_um,mean_mito_aspect_ratio,mean_mito_area_um,sum_mito
0,40PFOA_A3_P 1,OTHER,OTHER,7,4772.565857,216.054365,0.045270,131.0,1.388539,2.158164,2.416251,1.649270,477055
1,40PFOA_A3_P 1,OTHER,OTHER,12,5137.687242,402.004778,0.078246,236.0,1.506365,2.948833,2.549678,1.703410,678473
2,40PFOA_A3_P 1,OTHER,OTHER,13,3374.386622,349.383394,0.103540,186.0,1.714317,3.002719,2.562353,1.878405,615734
3,40PFOA_A3_P 1,OTHER,OTHER,14,6467.507102,549.941890,0.085032,218.0,2.002420,4.610408,2.724483,2.522669,902127
4,40PFOA_A3_P 1,OTHER,OTHER,15,4751.904700,543.888817,0.114457,118.0,1.524213,6.021641,2.662646,4.609227,950950
